# Plugging in a ZMART-driver

A driver connects the controller to one particular microscope. Writing one means completing a class, `ZmartDriver`, with one method per command. This notebook writes a driver for a simulated microscope that exists only in memory, validates it, installs it, and uses it like any other driver. Run the cells in order. The [README](README.md) specifies what every method must return.

## The two files

A driver is a folder with two files. `zmart_driver.json` holds the driver's name and the connection to the microscope; it is the file the controller is pointed at when the driver is installed. The simulated microscope needs no connection details, so its connection only names the instrument.

In [1]:
from pathlib import Path

Path("pretend_scope").mkdir(exist_ok=True)

In [2]:
%%writefile pretend_scope/zmart_driver.json
{
  "name": "pretend-scope",
  "driver": "zmart_driver.py",
  "connection": {
    "microscope": "pretend-01"
  }
}

Writing pretend_scope/zmart_driver.json


`zmart_driver.py` holds the class. Creating an instance opens the connection to the vendor software; here it only sets up a stage position and an exposure in memory. Each method carries out one command and returns `True` and the values, or `False` and a message giving the reason. The controller turns this into the `{"success": ..., "content": ...}` answer that a workflow receives. Two points are worth attention: `set_xyz` checks the travel limits before moving, and `acquire` reports where the image was saved and the stage position at which it was taken. That is how a workflow locates its images on any microscope.

In [3]:
%%writefile pretend_scope/zmart_driver.py
import json
from pathlib import Path

TRAVEL = {"x": [-1000.0, 1000.0], "y": [-1000.0, 1000.0], "z": [-100.0, 100.0]}   # micrometres from the origin
HALF_FIELD = 50.0                                                                   # half a field of view


class ZmartDriver:
    def __init__(self, connection):
        self.position = {"x": 0.0, "y": 0.0, "z": 0.0}
        self.settings = {"exposure_ms": 10.0}
        self.output_root = Path("pretend_scope_images")
        self.output_root.mkdir(exist_ok=True)

    def disconnect(self):
        pass

    def get_info(self):
        return True, "A pretend microscope in memory. exposure_ms is the exposure time in milliseconds. +z points up."

    def get_actuators(self):
        return True, (["motor"], ["motor"], ["motor"])

    def get_xyz(self, with_actuators):
        p = self.position
        canvas = (TRAVEL["x"][0] - HALF_FIELD, TRAVEL["x"][1] + HALF_FIELD,
                  TRAVEL["y"][0] - HALF_FIELD, TRAVEL["y"][1] + HALF_FIELD,
                  TRAVEL["z"][0], TRAVEL["z"][1])
        return True, (p["x"], p["y"], p["z"], "motor", "motor", "motor", canvas)

    def set_xyz(self, x, y, z, with_actuators):
        for axis, value in {"x": x, "y": y, "z": z}.items():
            low, high = TRAVEL[axis]
            if not low <= value <= high:
                return False, f"{axis} = {value} is outside the travel, {low} to {high}"
        self.position = {"x": x, "y": y, "z": z}
        return True, ("motor", "motor", "motor")

    def get_state(self):
        return True, (dict(self.settings), {"objective": "10x"})

    def set_state(self, changeable):
        for name in changeable:
            if name not in self.settings:
                return False, f"unknown setting {name}"
        self.settings.update(changeable)
        return True, dict(changeable)

    def get_acquisition_settings(self):
        return True, {"format": {"options": ["json"], "active": "json"}}

    def acquire(self, position_label, acquisition_settings):
        path = self.output_root / f"{position_label}.json"
        path.write_text(json.dumps({"position": self.position, "settings": self.settings}))
        p = self.position
        plane = {"path": str(path), "c": 0, "z": 0, "t": 0, "x_um": p["x"], "y_um": p["y"], "z_um": p["z"]}
        return True, ([str(path)], [plane])

    def get_procedures(self):
        return True, {}

    def run_procedure(self, procedure):
        return False, f"unknown procedure {procedure.get('name')}"

Writing pretend_scope/zmart_driver.py


## Validation

`validate_driver` connects, calls every `get_*` method, and compares each answer with what the controller expects. It returns the problems found, one sentence each. An empty list means the driver conforms. While writing a driver, this is the cycle to follow: write a method, validate, read the problems, repeat.

In [4]:
from zmart_controller import mic

mic.validate_driver("pretend_scope/zmart_driver.json")

[]

## Installation

Installing means pointing the controller at the driver's `zmart_driver.json`, once. The controller records it in its registry on this computer. From then on `get_instruments` lists it, and any Python session can connect to it by name.

In [5]:
mic.register_driver("pretend_scope/zmart_driver.json")
mic.get_instruments()

{'mock': {}, 'pretend-scope': {'microscope': 'pretend-01'}}

## Use

The simulated microscope now answers every command in the same form as the mock driver in Part 1. The "image" is a small text file recording where and how it was taken. The move outside the travel is answered with `success: False` and the message written by `set_xyz` above.

In [6]:
mic.connect("pretend-scope")
mic.set_xyz(100, 50, 0)
mic.acquire(position_label="A1")["content"]["files"]

['pretend_scope_images/A1.json']

In [7]:
mic.set_xyz(5000, 0, 0)

{'success': False,
 'content': 'x = 5000 is outside the travel, -1000.0 to 1000.0'}

In [8]:
mic.disconnect()

## Removal

This driver does not need to remain installed. For a real microscope, each method performs the same task by communicating with the vendor software, and the driver remains installed on the microscope computer.

In [9]:
mic.remove_driver("pretend-scope")
mic.get_instruments()

{'mock': {}}

---

MIT license. Thom de Hoog, Center for Microscopy and Image Analysis (ZMB), University of Zurich. thom.dehoog@zmb.uzh.ch, thomdehoog@gmail.com.